# 2 序列模型

## 2.1 理论计算题

给定一个字符序列 "ababc"，假设采用一阶马尔可夫模型（即 p(xt |xt−1)），使用拉普拉斯平滑（加1平滑）估计以下条件概率：
1. p(’a’ | ’b’)
2. p(’c’ | ’b’)

（词汇表为 {'a','b','c'}，计算时考虑所有可能转移，包括未出现的情况。）

In [5]:
# 2.1 理论计算题的计算过程

# 给定序列: "ababc"
# 转移对: (a,b), (b,a), (a,b), (b,c)

# 1. 统计转移次数
transitions = {
    'a': {'a': 0, 'b': 2, 'c': 0},
    'b': {'a': 1, 'b': 0, 'c': 1},
    'c': {'a': 0, 'b': 0, 'c': 0}
}

# 2. 词汇表大小
vocab_size = 3

# 3. 拉普拉斯平滑计算条件概率
def laplace_prob(prev_char, next_char):
    count_prev_next = transitions[prev_char][next_char]
    count_prev = sum(transitions[prev_char].values())
    return (count_prev_next + 1) / (count_prev + vocab_size)

# 计算问题1: p('a' | 'b')
p_a_given_b = laplace_prob('b', 'a')

# 计算问题2: p('c' | 'b')
p_c_given_b = laplace_prob('b', 'c')

print(f"1. p('a' | 'b') = {p_a_given_b:.4f}")
print(f"2. p('c' | 'b') = {p_c_given_b:.4f}")

# 详细计算过程
print("\n详细计算过程:")
print("转移统计:")
print("  a -> a: 0, a -> b: 2, a -> c: 0")
print("  b -> a: 1, b -> b: 0, b -> c: 1")
print("  c -> a: 0, c -> b: 0, c -> c: 0")
print(f"\n词汇表大小: {vocab_size}")
print(f"\n1. p('a' | 'b') = (1 + 1) / (2 + 3) = 2/5 = 0.4")
print(f"2. p('c' | 'b') = (1 + 1) / (2 + 3) = 2/5 = 0.4")

1. p('a' | 'b') = 0.4000
2. p('c' | 'b') = 0.4000

详细计算过程:
转移统计:
  a -> a: 0, a -> b: 2, a -> c: 0
  b -> a: 1, b -> b: 0, b -> c: 1
  c -> a: 0, c -> b: 0, c -> c: 0

词汇表大小: 3

1. p('a' | 'b') = (1 + 1) / (2 + 3) = 2/5 = 0.4
2. p('c' | 'b') = (1 + 1) / (2 + 3) = 2/5 = 0.4


## 2.2 编程题

编写一个函数 preprocess_text(text, n)，完成以下步骤：
1. 将文本转换为小写，去除标点符号（保留字母和空格）。
2. 按空格分词。
3. 构建词汇表（按出现频率排序，分配整数 ID，从 0 开始）。
4. 用滑动窗口生成长度为 n 的特征序列和对应的下一个词标签（用于自回归语言模型）。

返回词汇表字典和 (特征列表, 标签列表)。例如，输入 "The time machine" 和 n=2，应生成特征 [["the","time"], ["time","machine"]] 和标签 ["machine", None]（若无后续词则忽略）。

In [6]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理函数，用于自回归语言模型的数据准备
    
    参数:
        text (str): 输入文本
        n (int): 滑动窗口大小
    
    返回:
        vocab (dict): 词汇表字典，键为词，值为ID
        (features, labels): 特征列表和标签列表
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    # 使用正则表达式只保留字母和空格
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    # 按频率降序排序，频率相同则按字母顺序
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, count) in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签
    features = []
    labels = []
    
    # 只有当词语数量足够生成至少一个窗口时才继续
    if len(words) >= n:
        for i in range(len(words) - n + 1):
            # 特征：当前窗口的n个词
            feature = words[i:i+n]
            features.append(feature)
            
            # 标签：下一个词（如果存在），否则忽略（按题目说明）
            if i + n < len(words):
                label = words[i + n]
            else:
                label = None
            labels.append(label)
    
    return vocab, (features, labels)

# 测试示例
if __name__ == "__main__":
    # 测试输入
    test_text = "The time machine"
    test_n = 2
    
    vocab, (features, labels) = preprocess_text(test_text, test_n)
    
    print("测试示例：")
    print(f"输入文本: {test_text}")
    print(f"n = {test_n}")
    print(f"\n词汇表: {vocab}")
    print(f"特征: {features}")
    print(f"标签: {labels}")
    
    # 更多测试
    print("\n\n更多测试：")
    test_text2 = "Hello world! This is a test. Hello again."
    test_n2 = 3
    vocab2, (features2, labels2) = preprocess_text(test_text2, test_n2)
    print(f"输入文本: {test_text2}")
    print(f"n = {test_n2}")
    print(f"\n词汇表: {vocab2}")
    print(f"特征: {features2}")
    print(f"标签: {labels2}")

测试示例：
输入文本: The time machine
n = 2

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time'], ['time', 'machine']]
标签: ['machine', None]


更多测试：
输入文本: Hello world! This is a test. Hello again.
n = 3

词汇表: {'hello': 0, 'a': 1, 'again': 2, 'is': 3, 'test': 4, 'this': 5, 'world': 6}
特征: [['hello', 'world', 'this'], ['world', 'this', 'is'], ['this', 'is', 'a'], ['is', 'a', 'test'], ['a', 'test', 'hello'], ['test', 'hello', 'again']]
标签: ['is', 'a', 'test', 'hello', 'again', None]


# 3 循环神经网络

## 3.1 理论计算题

考虑一个线性 RNN（无偏置），定义为 ht = Whhht−1 + Whxxt，输出 ot = Wohht。假设损失函数为平方损失 L = 1/2 ∑T t=1(ot − yt)^2。推导损失对权重 Whh 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

### 3.1 理论推导

#### 1. 梯度表达式推导

**前向传播：**
- h_t = W_hh h_{t-1} + W_hx x_t
- o_t = W_o h_t
- 损失 L = (1/2) Σ_{t=1}^T (o_t - y_t)²

**反向传播（BPTT）：**
∂L/∂W_hh = Σ_{t=1}^T ∂L/∂h_t · ∂h_t/∂W_hh

其中，∂h_t/∂W_hh = Σ_{k=1}^t (Π_{i=k+1}^t W_hh) h_{k-1} ⊗ I

而 ∂L/∂h_t = ∂L/∂o_t · W_o + ∂L/∂h_{t+1} · W_hh

因此，完整的梯度表达式为：
∂L/∂W_hh = Σ_{t=1}^T [Σ_{k=1}^t (∂L/∂h_t) (W_hh)^{t-k}] h_{k-1}^T

#### 2. 梯度消失或爆炸的条件

- **梯度消失**：当 W_hh 的最大奇异值 σ_max < 1 时，梯度会随着时间步反向传播而指数级衰减。
- **梯度爆炸**：当 W_hh 的最大奇异值 σ_max > 1 时，梯度会随着时间步反向传播而指数级增长。
- **稳定**：当所有奇异值接近 1 时，梯度可以较好地传播。

对于使用 tanh 或 sigmoid 激活函数的非线性 RNN，由于这些函数的导数在 |x| 较大时趋近于 0，梯度消失问题更加严重。

## 3.2 编程题

实现一个简单的 RNN 单元的前向传播和单步反向传播（仅计算梯度，不更新）。给定输入 x_t（形状 (batch_size, input_size)）、上一隐藏状态 h_prev（形状 (batch_size, hidden_size)），以及权重 W_hx, W_hh, b_h，计算当前隐藏状态 h_t。同时实现反向传播，已知上游梯度 dh_next（即损失对 h_t 的梯度），计算 dx_t, dh_prev, dW_hx, dW_hh, db_h（使用 tanh 激活函数）。

In [7]:
import numpy as np

class SimpleRNNCell:
    def __init__(self, input_size, hidden_size):
        """
        初始化简单RNN单元
        
        参数:
            input_size: 输入特征维度
            hidden_size: 隐藏状态维度
        """
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # 初始化权重
        self.W_hx = np.random.randn(hidden_size, input_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.b_h = np.zeros((1, hidden_size))
        
        # 缓存用于反向传播
        self.cache = None
    
    def forward(self, x_t, h_prev):
        """
        前向传播
        
        参数:
            x_t: 当前输入，形状 (batch_size, input_size)
            h_prev: 上一时刻隐藏状态，形状 (batch_size, hidden_size)
            
        返回:
            h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        """
        # 计算线性部分
        linear = np.dot(x_t, self.W_hx.T) + np.dot(h_prev, self.W_hh.T) + self.b_h
        # 使用 tanh 激活函数
        h_t = np.tanh(linear)
        
        # 缓存用于反向传播
        self.cache = (x_t, h_prev, linear, h_t)
        
        return h_t
    
    def backward(self, dh_next):
        """
        单步反向传播
        
        参数:
            dh_next: 损失对当前隐藏状态 h_t 的梯度，形状 (batch_size, hidden_size)
            
        返回:
            dx_t: 损失对输入 x_t 的梯度，形状 (batch_size, input_size)
            dh_prev: 损失对上一隐藏状态 h_prev 的梯度，形状 (batch_size, hidden_size)
            dW_hx: 损失对权重 W_hx 的梯度，形状 (hidden_size, input_size)
            dW_hh: 损失对权重 W_hh 的梯度，形状 (hidden_size, hidden_size)
            db_h: 损失对偏置 b_h 的梯度，形状 (1, hidden_size)
        """
        x_t, h_prev, linear, h_t = self.cache
        batch_size = x_t.shape[0]
        
        # tanh 导数: 1 - tanh²(x)
        dtanh = dh_next * (1 - h_t ** 2)
        
        # 计算各梯度
        dx_t = np.dot(dtanh, self.W_hx)
        dh_prev = np.dot(dtanh, self.W_hh)
        dW_hx = np.dot(dtanh.T, x_t)
        dW_hh = np.dot(dtanh.T, h_prev)
        db_h = np.sum(dtanh, axis=0, keepdims=True)
        
        return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试
if __name__ == "__main__":
    # 超参数
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    # 初始化RNN单元
    rnn_cell = SimpleRNNCell(input_size, hidden_size)
    
    # 随机输入
    np.random.seed(42)
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    
    # 前向传播
    h_t = rnn_cell.forward(x_t, h_prev)
    print("前向传播结果 h_t:")
    print(h_t)
    print()
    
    # 假设的上游梯度
    dh_next = np.random.randn(batch_size, hidden_size)
    
    # 反向传播
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell.backward(dh_next)
    
    print("反向传播结果:")
    print(f"dx_t 形状: {dx_t.shape}")
    print(f"dh_prev 形状: {dh_prev.shape}")
    print(f"dW_hx 形状: {dW_hx.shape}")
    print(f"dW_hh 形状: {dW_hh.shape}")
    print(f"db_h 形状: {db_h.shape}")

前向传播结果 h_t:
[[ 0.00021829  0.004798   -0.02664376 -0.01265206]
 [-0.00708246 -0.01448722 -0.01300723  0.00352143]]

反向传播结果:
dx_t 形状: (2, 3)
dh_prev 形状: (2, 4)
dW_hx 形状: (4, 3)
dW_hh 形状: (4, 4)
db_h 形状: (1, 4)


# 4 高级循环神经网络

## 4.1 理论计算题

假设一个深度双向 RNN，有 L 层，每层隐藏单元数为 H，输入维度为 D，输出维度为 O（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

### 4.1 参数计算

#### 深度双向 RNN 参数总数

对于深度双向 RNN，每层包含前向和后向两个 RNN：

**第 1 层（输入层）：**
- 前向 RNN: $W_{hx}^{f} \in \mathbb{R}^{H \times D}$, $W_{hh}^{f} \in \mathbb{R}^{H \times H}$, $b_h^{f} \in \mathbb{R}^{H}$
- 后向 RNN: $W_{hx}^{b} \in \mathbb{R}^{H \times D}$, $W_{hh}^{b} \in \mathbb{R}^{H \times H}$, $b_h^{b} \in \mathbb{R}^{H}$
- 第 1 层总参数: $2 \times (H \times D + H \times H + H) = 2H(D + H + 1)$

**第 2 到 L 层（隐藏层）：**
- 每一层的输入维度是前一层前向和后向隐藏状态的拼接: $2H$
- 每层（前向+后向）参数: $2 \times (H \times 2H + H \times H + H) = 2H(3H + 1)$
- 第 2 到 L 层总参数: $(L-1) \times 2H(3H + 1)$

**输出层：**
- 输入维度: $2H$（最后一层前向和后向隐藏状态的拼接）
- 输出层参数: $O \times 2H + O = O(2H + 1)$

**总参数：**
$$
\text{总参数} = 2H(D + H + 1) + 2(L-1)H(3H + 1) + O(2H + 1)
$$

化简：
$$
\text{总参数} = 2HD + 2H^2 + 2H + 6(L-1)H^2 + 2(L-1)H + 2OH + O
$$
$$
\text{总参数} = 2HD + (2 + 6L - 6)H^2 + (2 + 2L - 2)H + 2OH + O
$$
$$
\text{总参数} = 2H(D + O) + 2(3L - 2)H^2 + 2LH + O
$$

## 4.2 编程题

实现一个双向 RNN 编码器，接收序列 X（形状 (seq_len, batch, input_dim)），使用 torch.nn.RNN 或手动实现。要求返回每个时间步的拼接后的前向和后向隐藏状态（形状 (seq_len, batch, 2*hidden_dim)），以及最终时间步的拼接隐藏状态（作为序列表示）。

In [8]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        """
        双向 RNN 编码器
        
        参数:
            input_dim: 输入特征维度
            hidden_dim: 隐藏状态维度（每个方向）
            num_layers: RNN 层数
            dropout: dropout 概率
        """
        super(BidirectionalRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 双向 RNN
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=False
        )
    
    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch_size, input_dim)
            
        返回:
            outputs: 每个时间步拼接后的前向和后向隐藏状态，形状 (seq_len, batch_size, 2*hidden_dim)
            final_hidden: 最终时间步的拼接隐藏状态，形状 (batch_size, 2*hidden_dim)
        """
        # RNN 前向传播
        outputs, hidden = self.rnn(X)
        # outputs 形状: (seq_len, batch_size, 2*hidden_dim)
        # hidden 形状: (2*num_layers, batch_size, hidden_dim)
        
        # 获取最后一层的前向和后向隐藏状态
        # 最后一层前向隐藏状态: hidden[-2, :, :]
        # 最后一层后向隐藏状态: hidden[-1, :, :]
        final_forward = hidden[-2, :, :]
        final_backward = hidden[-1, :, :]
        
        # 拼接前向和后向隐藏状态
        final_hidden = torch.cat([final_forward, final_backward], dim=1)
        # final_hidden 形状: (batch_size, 2*hidden_dim)
        
        return outputs, final_hidden

# 测试
if __name__ == "__main__":
    # 超参数
    seq_len = 10
    batch_size = 3
    input_dim = 5
    hidden_dim = 8
    num_layers = 2
    
    # 初始化模型
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers)
    
    # 随机输入
    X = torch.randn(seq_len, batch_size, input_dim)
    
    # 前向传播
    outputs, final_hidden = encoder(X)
    
    print("输入形状:", X.shape)
    print("输出 (所有时间步) 形状:", outputs.shape)
    print("最终隐藏状态形状:", final_hidden.shape)
    print()
    print("测试成功!")
    
    # 再测试一个简单例子
    print("\n简单测试 (单层):")
    simple_encoder = BidirectionalRNNEncoder(input_dim=4, hidden_dim=2, num_layers=1)
    simple_X = torch.tensor([[[1, 0, 0, 0], [0, 1, 0, 0]], 
                            [[0, 0, 1, 0], [0, 0, 0, 1]]], dtype=torch.float32)
    simple_outputs, simple_final = simple_encoder(simple_X)
    print("简单输入形状:", simple_X.shape)
    print("简单输出形状:", simple_outputs.shape)
    print("简单最终隐藏状态形状:", simple_final.shape)

输入形状: torch.Size([10, 3, 5])
输出 (所有时间步) 形状: torch.Size([10, 3, 16])
最终隐藏状态形状: torch.Size([3, 16])

测试成功!

简单测试 (单层):
简单输入形状: torch.Size([2, 2, 4])
简单输出形状: torch.Size([2, 2, 4])
简单最终隐藏状态形状: torch.Size([2, 4])


# 5 嵌入向量

## 5.1 理论计算题

在 Skip-gram 模型中，给定中心词 wc 和上下文词 wo，使用负采样（采样 K 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 vc, uo，负样本词向量为 unk，写出完整的目标函数。

### 5.1 Skip-gram 负采样损失推导

#### 1. 对数似然目标

在 Skip-gram 模型中，我们希望最大化给定中心词 $w_c$ 时上下文词 $w_o$ 出现的概率。

**正样本概率**：$P(w_o | w_c) = \sigma(u_o^\top v_c)$，其中 $\sigma(x) = 1/(1+e^{-x})$ 是 sigmoid 函数

**负样本**：从噪声分布 $P_n(w)$ 中采样 K 个负样本 $w_{n1}, w_{n2}, ..., w_{nK}$

**负样本概率**：对于每个负样本 $w_{nk}$，我们希望 $P(w_{nk} | w_c) = 1 - \sigma(u_{nk}^\top v_c)$

#### 2. 完整目标函数

对于一个 $(w_c, w_o)$ 训练样本，我们最大化对数似然：

$$
\mathcal{L} = \log \sigma(u_o^\top v_c) + \sum_{k=1}^K \mathbb{E}_{w_n \sim P_n(w)} [\log \sigma(-u_n^\top v_c)]
$$

等价的最小化负对数似然：

$$
\mathcal{L} = - \log \sigma(u_o^\top v_c) - \sum_{k=1}^K \log \sigma(-u_{nk}^\top v_c)
$$

#### 3. 噪声分布选择

通常使用**一元分布的幂次方**作为噪声分布：

$$
P_n(w) = \frac{U(w)^{3/4}}{Z}
$$

其中 $U(w)$ 是词 $w$ 在语料中的一元频率，$Z$ 是归一化常数。

这种分布的特点：
- 提高了稀有词的采样概率
- 降低了常见词的采样概率
- 相比均匀分布能更好地平衡训练样本

## 5.2 编程题

实现 CBOW 模型的前向传播和损失计算（不使用负采样，仅用完整 softmax）。给定一批上下文词的索引列表（每个样本有 context_size 个上下文词），词汇表大小 V，嵌入维度 d。输入权重矩阵 W（形状 (V, d)）和输出权重矩阵 W_out（形状 (d, V)）。计算平均上下文向量作为隐藏层，然后计算输出概率分布，并计算交叉熵损失（目标为中心词索引）。返回损失值。

In [9]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_indices, target_indices, W, W_out):
    """
    CBOW 模型前向传播和损失计算
    
    参数:
        context_indices: 上下文词索引，形状 (batch_size, context_size)
        target_indices: 中心词索引，形状 (batch_size)
        W: 输入权重矩阵，形状 (vocab_size, embedding_dim)
        W_out: 输出权重矩阵，形状 (embedding_dim, vocab_size)
    
    返回:
        loss: 交叉熵损失值（标量）
    """
    batch_size, context_size = context_indices.shape
    
    # 1. 获取上下文词的嵌入向量
    # context_embeds 形状: (batch_size, context_size, embedding_dim)
    context_embeds = W[context_indices]
    
    # 2. 计算平均上下文向量（隐藏层）
    # hidden 形状: (batch_size, embedding_dim)
    hidden = torch.mean(context_embeds, dim=1)
    
    # 3. 计算输出分数
    # scores 形状: (batch_size, vocab_size)
    scores = torch.matmul(hidden, W_out)
    
    # 4. 计算交叉熵损失
    loss = F.cross_entropy(scores, target_indices)
    
    return loss

# 测试
if __name__ == "__main__":
    # 超参数
    vocab_size = 10
    embedding_dim = 5
    batch_size = 3
    context_size = 4
    
    # 随机初始化权重
    torch.manual_seed(42)
    W = torch.randn(vocab_size, embedding_dim, requires_grad=True)
    W_out = torch.randn(embedding_dim, vocab_size, requires_grad=True)
    
    # 随机输入数据
    context_indices = torch.randint(0, vocab_size, (batch_size, context_size))
    target_indices = torch.randint(0, vocab_size, (batch_size,))
    
    print("上下文索引:")
    print(context_indices)
    print("\n目标索引:")
    print(target_indices)
    
    # 前向传播
    loss = cbow_forward_loss(context_indices, target_indices, W, W_out)
    
    print(f"\nCBOW 损失: {loss.item():.4f}")
    
    # 验证梯度计算
    loss.backward()
    print(f"\nW 的梯度范数: {W.grad.norm().item():.4f}")
    print(f"W_out 的梯度范数: {W_out.grad.norm().item():.4f}")

上下文索引:
tensor([[6, 0, 6, 8],
        [6, 8, 0, 6],
        [9, 0, 5, 9]])

目标索引:
tensor([5, 2, 0])

CBOW 损失: 2.3517

W 的梯度范数: 0.9324
W_out 的梯度范数: 0.5700


# 6 注意力机制

## 6.1 理论计算题

给定查询矩阵 Q ∈ R^2×4，键矩阵 K ∈ R^3×4，值矩阵 V ∈ R^3×5。计算缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤（先计算得分矩阵，再 softmax，再加权求和）。使用 score = QK^T / √d_k（d_k = 4）。

In [10]:
import numpy as np

# 为了演示，我们随机生成 Q, K, V 矩阵
np.random.seed(42)
Q = np.random.randn(2, 4)
K = np.random.randn(3, 4)
V = np.random.randn(3, 5)
d_k = 4

print("Q 矩阵:")
print(Q)
print("\nK 矩阵:")
print(K)
print("\nV 矩阵:")
print(V)

# 步骤 1: 计算得分矩阵 scores = Q @ K^T / sqrt(d_k)
scores = Q @ K.T / np.sqrt(d_k)
print("\n步骤 1 - 得分矩阵:")
print(scores)

# 步骤 2: 对每一行应用 softmax
def softmax(x):
    exps = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)

attention_weights = softmax(scores)
print("\n步骤 2 - 注意力权重:")
print(attention_weights)

# 步骤 3: 加权求和 V
output = attention_weights @ V
print("\n步骤 3 - 注意力输出:")
print(output)

Q 矩阵:
[[ 0.49671415 -0.1382643   0.64768854  1.52302986]
 [-0.23415337 -0.23413696  1.57921282  0.76743473]]

K 矩阵:
[[-0.46947439  0.54256004 -0.46341769 -0.46572975]
 [ 0.24196227 -1.91328024 -1.72491783 -0.56228753]
 [-1.01283112  0.31424733 -0.90802408 -1.4123037 ]]

V 矩阵:
[[ 1.46564877 -0.2257763   0.0675282  -1.42474819 -0.54438272]
 [ 0.11092259 -1.15099358  0.37569802 -0.60063869 -0.29169375]
 [-0.60170661  1.85227818 -0.01349722 -1.05771093  0.82254491]]

步骤 1 - 得分矩阵:
[[-0.65884095 -0.79443288 -1.64281711]
 [-0.55317835 -1.382109   -1.17711663]]

步骤 2 - 注意力权重:
[[0.44503374 0.38860296 0.1663633 ]
 [0.50701047 0.22131809 0.27167143]]

步骤 3 - 注意力输出:
[[ 0.5952661  -0.23960648  0.17380425 -1.04343526 -0.21878045]
 [ 0.60418195  0.13400442  0.11371947 -1.1426443  -0.11710289]]


## 6.2 编程题

实现多头注意力（Multi-Head Attention）的前向传播，假设 num_heads=2，d_model=4。给定输入 X（形状 (seq_len, batch, d_model)），分别线性投影得到 Q, K, V（每个头的维度 d_k = d_v = d_model/num_heads）。对每个头计算缩放点积注意力，然后将所有头的输出拼接并经过最终线性层。返回输出（形状与输入相同）。

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, X):
        """
        参数:
            X: 输入序列 (seq_len, batch, d_model)
        
        返回:
            output: 注意力输出 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        
        # 步骤 1: 线性投影 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 步骤 2: 分割成多个头
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2).transpose(1, 2)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2).transpose(1, 2)
        V = V.view(seq_len, batch, self.num_heads, self.d_k).transpose(0, 2).transpose(1, 2)
        # 现在形状: (num_heads, batch, seq_len, d_k)
        
        # 步骤 3: 缩放点积注意力
        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)  # (num_heads, batch, seq_len, seq_len)
        attention_weights = F.softmax(scores, dim=-1)
        head_outputs = attention_weights @ V  # (num_heads, batch, seq_len, d_k)
        
        # 步骤 4: 拼接头的输出
        head_outputs = head_outputs.transpose(1, 2).transpose(0, 2).contiguous()
        head_outputs = head_outputs.view(seq_len, batch, self.num_heads * self.d_k)
        
        # 步骤 5: 最终线性层
        output = self.W_o(head_outputs)
        
        return output

# 测试
if __name__ == "__main__":
    # 超参数
    seq_len = 5
    batch_size = 2
    d_model = 4
    num_heads = 2
    
    # 初始化模型
    torch.manual_seed(42)
    mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
    
    # 随机输入
    X = torch.randn(seq_len, batch_size, d_model)
    print("输入形状:", X.shape)
    
    # 前向传播
    output = mha(X)
    print("输出形状:", output.shape)
    
    assert output.shape == X.shape, "输出形状应该与输入相同!"
    print("\n测试成功!")

输入形状: torch.Size([5, 2, 4])
输出形状: torch.Size([5, 2, 4])

测试成功!
